# Ball and Beam Simulation

Simulating a tiltiing beam balancing a ball at the middle position.

Importing packages

In [144]:
try: 
    from jax import config
    config.update("jax_enable_x64", True)
    import time
    import math
    import jax.numpy as jnp
    from meshcat import Visualizer
    import meshcat.geometry as mc_geom
    import meshcat.transformations as mc_trans
    
    # local imports
    from pid import PIDController



    print('Imported packages.')
except Exception as e:
    print('Importing packages failed:')
    print(e)



Imported packages.


Integrator and constants.

In [145]:
R = 0.01 # meter
m_ball = 0.05 # kg
m_beam = 0.5 # kg
l_beam = 0.3 # meter
g = 9.81 # m/s^2
D = 100 # friction coefficient at the edges 
q_init = [0., 0., 0., jnp.pi / 36] # ball pos[m], ball angle[rad], beam pos[m], beam angle[rad]
qdot_init = [0., 0., 0., 0.] # ball v[m/s], ball omega[rad/s], beam v(always 0), beam omega[rad/s]
M = jnp.diag(jnp.array([m_ball, m_beam]))
I = jnp.diag(jnp.array([2/5 * m_ball * R * R, m_beam * l_beam * l_beam /12]))

controller = PIDController(1, 0, 0, 0)
servo_controller = PIDController(1, 0, 0, 0)

# Runge-Kutta 4
def rk4_step(f, x, dt):
    """
        Input:
            f(x) - function to be integrated
            x - current state
            dt - time step 
        Output: 
            x[t+dt] 
    """
    q, qdot = jnp.split(x, 2)
    pid_out = controller.calculate(-q[0], dt)
    # servo controller(internal to the servo)
    servo_controller.setpoint = pid_out
    u = servo_controller.calculate(q[3], dt)
    k1 = f(x, u)
    k2 = f(x + dt*k1/2, u)
    k3 = f(x + dt*k2/2, u)
    k4 = f(x + dt*k3, u)
    xdot = x + 1/6*(k1+2*k2+2*k3+k4)*dt
    return xdot 

The following function computes the dynamics of the system each timestep.

In [146]:
def dyn_step(x, u):
    """
        Input: 
            state = x = [q, qdot] 
        Output: 
            xdot = [qdot, qddot]
    """
    q, qdot = jnp.split(x, 2)
    v_ball, omega_ball, v_beam, omega_beam = qdot
    # ball control(real-time simulation)
    ball_xddot = - 5 / 7 * g * math.sin(q[3])
    ball_thetaddot = ball_xddot / R
    if (q[0] - R <= -l_beam / 2 and ball_xddot < 0) or (q[0] + R >= l_beam / 2 and ball_xddot > 0):
        ball_xddot = 0 
        ball_thetaddot = -jnp.sign(omega_ball) * D # Damping coeff 
        v_ball = 0
    omega_beam = u
    # check beam max tilt
    if abs(q[3]) > math.pi / 6:
        omega_beam = 0 
    qdot = jnp.array([v_ball, omega_ball, v_beam, omega_beam])
    qddot = jnp.array([ball_xddot, ball_thetaddot, 0., 0.])
    return jnp.hstack(jnp.array([qdot, qddot]))

Initialize the visualizer.

In [147]:
class Viewer1D(Visualizer):
    def __init__(self, R, l_beam, x_init) -> None:
        Visualizer.__init__(self)
        self._ball_R = R
        self._beam_thickness = 0.01
        self._beam = self["beam"]
        self._ball= self["end_effector"]
        self._ball.set_object(
            mc_geom.Sphere(radius=R),
            mc_geom.MeshLambertMaterial(
                    #color=0x0000ff,
                    # opacity=0.5,
                    reflectivity=0.8,
                    map=mc_geom.ImageTexture(image=mc_geom.PngImage.from_file('./BeachBallColor.jpg'))
                    )
        )
        self._beam.set_object(
            mc_geom.Box([self._beam_thickness, l_beam, self._beam_thickness]),
            mc_geom.MeshLambertMaterial(
                    color=0x00ff00,
                    # opacity=0.5,
                    reflectivity=0.8,
            )
        )
        self.render(x_init)
    def render(self, x):
            x_ball, theta_ball, _, theta_beam = x
            _T_ball = mc_trans.compose_matrix(
                            translate=[0.,x_ball * math.cos(theta_beam) + (self._ball_R + self._beam_thickness / 2) * math.cos(theta_beam + math.pi/2),x_ball * math.sin(theta_beam) + (self._ball_R + self._beam_thickness / 2) * math.sin(theta_beam + math.pi/2)], 
                            angles=[0.,theta_ball,math.pi/2]
                )
            _T_beam = mc_trans.compose_matrix(
                            translate=[0.,0,0], 
                            angles=[theta_beam, 0.,0.]
                )
            self._ball.set_transform(_T_ball)
            self._beam.set_transform(_T_beam)

viewer = Viewer1D(R, l_beam, q_init)
viewer.jupyter_cell()

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/


## Animation

Using RK4 for integration to propagate.

In [148]:
tf = 20
dt = 1e-2
N = int(tf/dt)
x0 = jnp.array(q_init + qdot_init)
for k in range(N):
    x0 = rk4_step(dyn_step, x0, dt)
    q, qdot = jnp.split(x0, 2)
    viewer.render(q)
    time.sleep(dt)